# Tabelas de referência (ref_*) — Bronze → Silver

Análise e tratamento das 10 tabelas de dicionário/de-para do Novo Caged que hoje vivem na bronze (`ref_regiao`, `ref_uf`, `ref_municipio`, `ref_secao`, `ref_subclasse`, `ref_categoria`, `ref_cbo_ocupacao`, `ref_grau_instrucao`, `ref_tipo_movimentacao`, `ref_tipo_estabelecimento`), criando as versões tratadas em `workspace.silver`.

## Roteiro

1. **Perfil geral** — linhas, vazios, espaços excedentes, códigos duplicados e códigos fora do padrão numérico
2. **Domínios e amostras** — conteúdo completo das tabelas pequenas, amostras das grandes e descrições ambíguas (mesma descrição para códigos diferentes)
3. **Conclusões** — tratamentos identificados
4. **Tratamento e carga** — tipagem do código, trim, criação protegida com comentários e `INSERT OVERWRITE` na silver
5. **Validação** — contagens bronze × silver, nulos pós-conversão e duplicidades

In [0]:
# 1) Perfil geral das tabelas de referência na bronze
import pandas as pd

# tabela -> tipo pretendido para o código na silver
refs = {
    'ref_regiao': 'SMALLINT',
    'ref_uf': 'SMALLINT',
    'ref_municipio': 'INT',
    'ref_secao': 'STRING',
    'ref_subclasse': 'INT',
    'ref_categoria': 'SMALLINT',
    'ref_cbo_ocupacao': 'INT',
    'ref_grau_instrucao': 'SMALLINT',
    'ref_tipo_movimentacao': 'SMALLINT',
    'ref_tipo_estabelecimento': 'SMALLINT',
}

dados = []
for t, tipo in refs.items():
    src = f"workspace.bronze.{t}"
    r = spark.sql(f"""
      SELECT count(*) AS linhas,
             count(DISTINCT trim(codigo)) AS codigos_distintos,
             sum(CASE WHEN trim(codigo) IS NULL OR trim(codigo) = '' THEN 1 ELSE 0 END) AS codigo_vazio,
             sum(CASE WHEN trim(descricao) IS NULL OR trim(descricao) = '' THEN 1 ELSE 0 END) AS descricao_vazia,
             sum(CASE WHEN codigo IS NOT NULL AND codigo != trim(codigo) THEN 1 ELSE 0 END) AS codigo_espacos,
             sum(CASE WHEN descricao IS NOT NULL AND descricao != trim(descricao) THEN 1 ELSE 0 END) AS descricao_espacos,
             count(DISTINCT trim(descricao)) AS descricoes_distintas
      FROM {src}
    """).collect()[0]
    nao_num = spark.sql(
        f"SELECT count(*) AS n FROM {src} WHERE trim(codigo) NOT RLIKE '^-?[0-9]+$'"
    ).collect()[0]['n'] if tipo != 'STRING' else 0
    exemplos = [str(x['codigo']) for x in spark.sql(
        f"SELECT DISTINCT trim(codigo) AS codigo FROM {src} WHERE trim(codigo) NOT RLIKE '^-?[0-9]+$' LIMIT 5"
    ).collect()] if nao_num else []
    dados.append({
        'tabela': t,
        'tipo_codigo_silver': tipo,
        'linhas': r['linhas'],
        'codigos_distintos': r['codigos_distintos'],
        'codigos_duplicados': r['linhas'] - r['codigos_distintos'],
        'codigo_vazio': r['codigo_vazio'],
        'descricao_vazia': r['descricao_vazia'],
        'com_espacos': r['codigo_espacos'] + r['descricao_espacos'],
        'descricoes_repetidas': r['linhas'] - r['descricoes_distintas'],
        'codigos_nao_numericos': nao_num,
        'exemplos_nao_numericos': '; '.join(exemplos),
    })
display(pd.DataFrame(dados))

In [0]:
# 2) Domínios das tabelas pequenas, amostras das grandes e descrições ambíguas
# Tabelas pequenas: conteúdo completo (union em uma única visualização)
pequenas = ['ref_regiao', 'ref_uf', 'ref_secao', 'ref_categoria',
            'ref_grau_instrucao', 'ref_tipo_movimentacao', 'ref_tipo_estabelecimento']
uniao = " UNION ALL ".join(
    f"(SELECT '{t}' AS tabela, trim(codigo) AS codigo, trim(descricao) AS descricao FROM workspace.bronze.{t})"
    for t in pequenas)
display(spark.sql(f"{uniao} ORDER BY tabela, codigo"))

# Tabelas grandes: amostra de 5 linhas
for t in ['ref_municipio', 'ref_subclasse', 'ref_cbo_ocupacao']:
    print(f"\n--- amostra: {t} ---")
    display(spark.sql(f"SELECT trim(codigo) AS codigo, trim(descricao) AS descricao FROM workspace.bronze.{t} LIMIT 5"))

# Descrições ambíguas: mesma descrição para mais de um código
amb = " UNION ALL ".join(f"""(
    SELECT '{t}' AS tabela, trim(descricao) AS descricao,
           count(DISTINCT trim(codigo)) AS qtd_codigos,
           count(*) AS linhas
    FROM workspace.bronze.{t}
    GROUP BY 1, 2
    HAVING count(DISTINCT trim(codigo)) > 1
  )""" for t in refs)
display(spark.sql(f"{amb} ORDER BY qtd_codigos DESC, tabela LIMIT 20"))

## 3) Conclusões — tratamentos para a Silver

**Qualidade das 10 tabelas na bronze: alta.** Nenhuma tabela apresenta código ou descrição vazios, espaços excedentes, códigos duplicados ou códigos fora do padrão (numéricos em todas, exceto `ref_secao` cujo código é alfabético por natureza: A–U e Z).

### Achados (sem necessidade de correção, apenas documentação)

* **Descrições com mais de um código (legítimas, 2 casos):** `Rs-Pinto Bandeira` em `ref_municipio` (município historicamente dividido entre dois códigos) e `Guarda Portuário` em `ref_cbo_ocupacao`. O dicionário oficial traz ambos — **não deduplicar**.
* **Grafias inconsistentes do dicionário oficial** (preservadas por fidelidade à fonte): `Minas gerais` com caixa baixa, variação `Não identificado` × `Não Identificado`, acentuação atípica (`AqÜIcultura`), municípios com prefixo da UF (`Pe-Caruaru`, `Sp-Sao Paulo`).
* **Cobertura:** o dicionário não contém alguns códigos usados nos movimentos (`regiao=99`, 42 CBOs, `subclasse=8630505`, `município=510183`, `tipoestabelecimento=-1`) — a silver `novo_caged_movimentacoes` já lida com isso deixando a descrição `NULL`; o dicionário é mantido **puro**, sem inserção de códigos artificiais.

### Tratamentos aplicados na carga

| Tratamento | Detalhe |
| --- | --- |
| **Tipagem do código** | `SMALLINT` (região, UF, categoria, grau, tipo movimentação, tipo estabelecimento), `INT` (município, subclasse, CBO), `STRING` (seção) — via `try_cast` |
| **Limpeza** | `trim` em código e descrição (defensivo — dados já estão limpos) |
| **Criação protegida** | Tabela criada apenas se não existir (guarda com `SHOW TABLES`), com comentários de tabela e coluna em pt-BR |
| **Carga idempotente** | `INSERT OVERWRITE` — recarrega integralmente a cada execução (atualiza quando o dicionário na bronze mudar, preservando schema e comentários) |

In [0]:
# 4) Tratamento e carga: criação protegida + INSERT OVERWRITE na silver
# Conversões: código tipado (SMALLINT/INT; seção permanece STRING), trim em ambos
REFS_META = {
    'ref_regiao': ('SMALLINT', 'Regiões geográficas (dicionário do Novo Caged)', 'Código da região', 'Nome da região'),
    'ref_uf': ('SMALLINT', 'Unidades da Federação (dicionário do Novo Caged)', 'Código da UF', 'Nome da UF'),
    'ref_municipio': ('INT', 'Municípios (dicionário do Novo Caged)', 'Código IBGE do município', 'Nome do município (com prefixo da UF)'),
    'ref_secao': ('STRING', 'Seções CNAE (dicionário do Novo Caged)', 'Código da seção CNAE', 'Nome da seção CNAE'),
    'ref_subclasse': ('INT', 'Subclasses CNAE (dicionário do Novo Caged)', 'Código da subclasse CNAE', 'Nome da subclasse CNAE'),
    'ref_categoria': ('SMALLINT', 'Categorias de trabalhador (dicionário do Novo Caged)', 'Código da categoria', 'Descrição da categoria'),
    'ref_cbo_ocupacao': ('INT', 'Ocupações CBO 2002 (dicionário do Novo Caged)', 'Código CBO 2002', 'Descrição da ocupação'),
    'ref_grau_instrucao': ('SMALLINT', 'Graus de instrução (dicionário do Novo Caged)', 'Código do grau de instrução', 'Descrição do grau de instrução'),
    'ref_tipo_movimentacao': ('SMALLINT', 'Tipos de movimentação (dicionário do Novo Caged)', 'Código do tipo de movimentação', 'Descrição do tipo de movimentação'),
    'ref_tipo_estabelecimento': ('SMALLINT', 'Tipos de estabelecimento (dicionário do Novo Caged)', 'Código do tipo de estabelecimento', 'Descrição do tipo de estabelecimento'),
}

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

for t, (tipo, coment_tab, coment_cod, coment_desc) in REFS_META.items():
    src = f"workspace.bronze.{t}"
    dst = f"workspace.silver.{t}"
    existe = spark.sql(f"SHOW TABLES IN workspace.silver LIKE '{t}'").count() > 0
    if not existe:
        spark.sql(f"""
          CREATE TABLE {dst} (
            codigo {tipo} COMMENT '{coment_cod}',
            descricao STRING COMMENT '{coment_desc}'
          )
          USING DELTA
          COMMENT '{coment_tab}'
        """)
    # INSERT OVERWRITE: recarrega integralmente a cada execução (idempotente,
    # atualiza quando o dicionário na bronze mudar, preserva schema e comentários)
    spark.sql(f"""
      INSERT OVERWRITE {dst}
      SELECT try_cast(trim(codigo) AS {tipo}) AS codigo,
             trim(descricao) AS descricao
      FROM {src}
    """)
    print(f"{t}: carregada (código {tipo})")
print("\nCarga das 10 tabelas ref_* concluída.")

In [0]:
# 5) Validação: bronze × silver, nulos pós-conversão e duplicidades
import pandas as pd

dados = []
for t in REFS_META:
    src = f"workspace.bronze.{t}"
    dst = f"workspace.silver.{t}"
    b = spark.sql(f"SELECT count(*) AS n FROM {src}").collect()[0]['n']
    r = spark.sql(f"""
      SELECT count(*) AS linhas,
             count(DISTINCT codigo) AS codigos_distintos,
             sum(CASE WHEN codigo IS NULL THEN 1 ELSE 0 END) AS codigo_nulo,
             sum(CASE WHEN descricao IS NULL OR trim(descricao) = '' THEN 1 ELSE 0 END) AS descricao_nula
      FROM {dst}
    """).collect()[0]
    ok = (r['linhas'] == b and r['linhas'] == r['codigos_distintos']
          and r['codigo_nulo'] == 0 and r['descricao_nula'] == 0)
    dados.append({
        'tabela': t,
        'linhas_bronze': b,
        'linhas_silver': r['linhas'],
        'codigos_distintos': r['codigos_distintos'],
        'codigo_nulo': r['codigo_nulo'],
        'descricao_nula': r['descricao_nula'],
        'status': 'OK' if ok else 'VERIFICAR',
    })
display(pd.DataFrame(dados))

# Amostras da silver (código já tipado)
display(spark.sql("SELECT * FROM workspace.silver.ref_regiao ORDER BY codigo"))
display(spark.sql("SELECT * FROM workspace.silver.ref_tipo_movimentacao ORDER BY codigo"))